# 🤗 M1MLOP — Jour 4 : Natural Language Processing (NLP)
**École IT — Master 1 — Unité 3 BigData — Semaine 11**  
**Amadou MAIGA**

---

## 🗂️ Table des matières
1. [Objectifs et contexte](#1)
2. [Fondements NLP — Tokenisation](#2)
3. [Embeddings — Convertir texte en vecteurs](#3)
4. [TP 4.1 — Tokenisation et preprocessing NLP](#4)
5. [Transformers — BERT vs GPT](#5)
6. [TP 4.2 — Fine-tuning BERT pour Sentiment Analysis](#6)
7. [Named Entity Recognition (NER)](#7)
8. [TP 4.3 — NER avec spaCy](#8)
9. [Question Answering (QA)](#9)
10. [TP 4.4 — API Flask pour modèle NLP](#10)
11. [MLOps appliqué au NLP](#11)
12. [Résumé et bilan du Jour 4](#12)

---

> 📌 **Prérequis** : Jours 1-3 complétés (MLflow, Airflow, Monitoring)  
> ⏱️ **Durée estimée** : 7h (matinée + après-midi)

<a id='1'></a>
## 1. 🎯 Objectifs du Jour 4

À la fin de cette journée, tu seras capable de :

- ✅ Comprendre les fondements du NLP : **tokenisation**, **preprocessing**, **embeddings**
- ✅ Utiliser les **Transformers modernes** (BERT, GPT) avec Hugging Face
- ✅ **Fine-tuner** un modèle pré-entraîné pour une tâche spécifique
- ✅ Implémenter des applications NLP : **sentiment analysis**, **NER**, **classification**
- ✅ Appliquer les **MLOps** aux modèles de langage

### 🔗 Lien avec les jours précédents

| Jours 1–3 | Jour 4 |
|---|---|
| Modèles ML classiques (iris, tabular) | Modèles sur **données textuelles** |
| Drift sur features numériques | Drift sur **distributions de sentiments** |
| MLflow tracking (accuracy, F1) | MLflow tracking (**loss, perplexity, F1 NLP**) |
| API Flask avec RandomForest | API Flask avec **BERT/DistilBERT** |

<a id='2'></a>
## 2. 📝 Fondements NLP — Tokenisation

En ML classique, les données sont déjà des **nombres**. En NLP, on doit convertir du **texte en nombres** que les modèles peuvent comprendre.

### Pipeline NLP complet

```
Texte brut
    │
    ▼
Preprocessing (lowercase, ponctuation, stopwords)
    │
    ▼
Tokenisation (découper en tokens)
    │
    ▼
Embeddings (tokens → vecteurs numériques)
    │
    ▼
Modèle (BERT, GPT, etc.)
    │
    ▼
Prédiction (sentiment, entité, classe...)
```

### Types de tokenisation

| Type | Exemple | Utilisé par |
|---|---|---|
| **Word-level** | `["Bonjour", ",", "comment"]` | NLTK, spaCy |
| **Subword (BPE)** | `["Bon", "jour", ","]` | BERT, GPT |
| **Character-level** | `['B','o','n','j','o','u','r']` | modèles anciens |

> 💬 **Google Translate** : La tokenisation est critique pour la traduction. Une mauvaise tokenisation = une mauvaise traduction.

In [ ]:
# ============================================================
# 📦 INSTALLATION DES DÉPENDANCES JOUR 4
# ============================================================
import sys

# Dépendances de base (sans GPU requis)
!{sys.executable} -m pip install transformers datasets scikit-learn pandas numpy matplotlib --quiet
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu --quiet

print("✅ Installation terminée !")
print()
print("ℹ️  Pour spaCy + modèle français :")
print("   pip install spacy")
print("   python -m spacy download fr_core_news_sm")
print()
print("💡 Note : Les modèles Hugging Face seront téléchargés à la première utilisation.")
print("   (DistilBERT ≈ 250MB, téléchargé une seule fois dans ~/.cache/huggingface)")

In [ ]:
# ============================================================
# 📚 IMPORTS GLOBAUX
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import time
import re
import json
import os
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✅ Imports de base OK")

# Imports NLP
try:
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModel,
        AutoModelForSequenceClassification,
        pipeline,
        TrainingArguments,
        Trainer
    )
    from datasets import Dataset
    print(f"✅ PyTorch version  : {torch.__version__}")
    print(f"✅ Transformers OK")
    print(f"   GPU disponible  : {torch.cuda.is_available()} (CPU utilisé si False)")
    TRANSFORMERS_AVAILABLE = True
except ImportError as e:
    print(f"⚠️  Transformers non disponible : {e}")
    print("   Lance d'abord la cellule d'installation !")
    TRANSFORMERS_AVAILABLE = False

<a id='4'></a>
## 3. 🧪 TP 4.1 — Tokenisation et Preprocessing NLP

### Objectif
Implémenter un **pipeline de preprocessing NLP complet** sans dépendances lourdes, puis comparer avec un tokenizer moderne (Hugging Face).

In [ ]:
# ============================================================
# TP 4.1 — PREPROCESSING NLP MANUEL
# ============================================================

# Dataset de phrases pour les TPs
SAMPLE_TEXTS = [
    "Excellent produit, très satisfait de mon achat ! Je recommande vivement.",
    "Mauvaise qualité, complètement déçu. Ne fonctionne pas du tout.",
    "Service client irréprochable, livraison rapide. Parfait !",
    "Produit défectueux reçu. Remboursement refusé. Scandaleux !",
    "Rapport qualité-prix correct. Pas exceptionnel mais satisfaisant.",
    "Incroyable ! Meilleur achat de l'année, je suis ravi.",
    "Délai de livraison trop long, emballage abîmé. Déçu.",
    "Fonctionne exactement comme décrit. Très bonne surprise.",
    "Commande annulée sans explication. Service désastreux.",
    "Qualité premium, finition parfaite. J'adore ce produit !",
    "Rien ne correspond à la description. Arnaque complète.",
    "Très bon produit pour le prix. Je recommande à tous.",
]

LABELS = [1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1]  # 1=positif, 0=négatif

# Stopwords français simplifiés
STOPWORDS_FR = {
    'le','la','les','de','du','des','un','une','et','en','à','au','aux',
    'je','tu','il','elle','nous','vous','ils','elles','me','mon','ma','mes',
    'ce','cet','cette','ces','se','sa','son','ses','qui','que','qu','ne',
    'pas','plus','très','bien','tout','tous','toute','toutes','est','sont',
    'par','pour','sur','dans','avec','ou','mais','donc','car','ni','si'
}

def preprocess_text(text: str,
                    lowercase: bool = True,
                    remove_punctuation: bool = True,
                    remove_stopwords: bool = True) -> dict:
    """
    Pipeline de preprocessing NLP complet.

    Étapes :
    1. Lowercase
    2. Suppression de la ponctuation
    3. Tokenisation (split par espace)
    4. Suppression des stopwords
    """
    original = text

    # Étape 1 : Lowercase
    if lowercase:
        text = text.lower()

    # Étape 2 : Suppression ponctuation et caractères spéciaux
    if remove_punctuation:
        text = re.sub(r"[^\w\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()

    # Étape 3 : Tokenisation word-level
    tokens = text.split()

    # Étape 4 : Suppression stopwords
    if remove_stopwords:
        tokens_no_stop = [t for t in tokens if t not in STOPWORDS_FR and len(t) > 2]
    else:
        tokens_no_stop = tokens

    return {
        "original"         : original,
        "cleaned"          : text,
        "tokens"           : tokens,
        "tokens_no_stop"   : tokens_no_stop,
        "n_tokens_before"  : len(tokens),
        "n_tokens_after"   : len(tokens_no_stop),
        "reduction_pct"    : round((1 - len(tokens_no_stop)/max(len(tokens),1)) * 100, 1)
    }

# Tester sur les premières phrases
print("📋 Pipeline de Preprocessing NLP — TP 4.1")
print("=" * 70)

for i, text in enumerate(SAMPLE_TEXTS[:4]):
    result = preprocess_text(text)
    label_str = "😊 POSITIF" if LABELS[i] == 1 else "😞 NÉGATIF"
    print(f"\n  [{label_str}] Texte #{i+1}")
    print(f"  Original     : {result['original'][:60]}...")
    print(f"  Tokens bruts : {result['tokens'][:8]}")
    print(f"  Sans stops   : {result['tokens_no_stop'][:8]}")
    print(f"  Réduction    : {result['n_tokens_before']} → {result['n_tokens_after']} tokens ({result['reduction_pct']}% de réduction)")

In [ ]:
# ============================================================
# BAG OF WORDS — Représentation vectorielle simple
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

print("📊 Bag of Words vs TF-IDF — Représentations vectorielles")
print("=" * 65)

texts_clean = [" ".join(preprocess_text(t)["tokens_no_stop"]) for t in SAMPLE_TEXTS]

# BoW
bow_vectorizer = CountVectorizer(max_features=50)
X_bow = bow_vectorizer.fit_transform(texts_clean).toarray()

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=50, ngram_range=(1,2))
X_tfidf = tfidf_vectorizer.fit_transform(texts_clean).toarray()

print(f"  Vocabulaire BoW       : {len(bow_vectorizer.vocabulary_)} mots")
print(f"  Vocabulaire TF-IDF    : {len(tfidf_vectorizer.vocabulary_)} n-grams")
print(f"  Shape matrice BoW     : {X_bow.shape}")
print(f"  Shape matrice TF-IDF  : {X_tfidf.shape}")
print()

# Classifier avec BoW + Naive Bayes
labels = np.array(LABELS)
nb_bow = MultinomialNB()
nb_bow.fit(X_bow, labels)
preds_bow = nb_bow.predict(X_bow)
acc_bow = accuracy_score(labels, preds_bow)

# Classifier avec TF-IDF + Naive Bayes
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_tfidf, labels)
preds_tfidf = nb_tfidf.predict(X_tfidf)
acc_tfidf = accuracy_score(labels, preds_tfidf)

print(f"  Accuracy BoW + NaiveBayes   : {acc_bow:.4f}")
print(f"  Accuracy TF-IDF + NaiveBayes: {acc_tfidf:.4f}")
print()
print("  💡 Ces approches classiques seront dépassées par BERT (section suivante)")

# Top mots TF-IDF
feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()
mean_tfidf = X_tfidf.mean(axis=0)
top_features = sorted(zip(mean_tfidf, feature_names_tfidf), reverse=True)[:10]
print()
print("  🔝 Top 10 mots/n-grams TF-IDF :")
for score, word in top_features:
    print(f"     {word:25s} {score:.4f}")

In [ ]:
# ============================================================
# TOKENIZER HUGGING FACE — Subword Tokenization
# ============================================================

if TRANSFORMERS_AVAILABLE:
    print("🤗 Tokenizer Hugging Face — DistilBERT")
    print("=" * 65)

    # Charger le tokenizer (téléchargement ~250MB la 1ère fois)
    MODEL_NAME = "distilbert-base-multilingual-cased"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print(f"  Modèle         : {MODEL_NAME}")
    print(f"  Taille vocab   : {tokenizer.vocab_size:,} tokens")
    print()

    # Comparer tokenisation word-level vs subword
    test_phrases = [
        "Excellent produit, très satisfait !",
        "Machine learning et intelligence artificielle",
        "anticonstitutionnellement"
    ]

    for phrase in test_phrases:
        word_tokens = phrase.lower().split()
        hf_tokens   = tokenizer.tokenize(phrase)
        ids         = tokenizer.encode(phrase)

        print(f"  Phrase   : '{phrase}'")
        print(f"  Word-level ({len(word_tokens)} tokens): {word_tokens}")
        print(f"  HF Subword ({len(hf_tokens)} tokens): {hf_tokens}")
        print(f"  IDs        : {ids[:10]}...")
        print()

    # Tokeniser tout le dataset
    encoded = tokenizer(
        SAMPLE_TEXTS,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors='pt'
    )

    print(f"  Dataset tokenisé :")
    print(f"     input_ids shape      : {encoded['input_ids'].shape}")
    print(f"     attention_mask shape : {encoded['attention_mask'].shape}")
    print()
    print("  💡 attention_mask = 1 pour vrais tokens, 0 pour padding")
else:
    print("⚠️  Transformers non disponible — installe les dépendances d'abord")

<a id='3'></a>
## 4. 🔢 Embeddings — Convertir Texte en Vecteurs

Les **embeddings** sont des représentations numériques denses de tokens ou phrases.

### Propriétés des embeddings sémantiques

```
Espace d'embeddings (768 dimensions dans BERT) :

  "Chat" ←──── proche ────→ "Chien"   (même catégorie : animaux)
  "Chat" ←──── loin ──────→ "Voiture" (catégories différentes)
  "Bonjour" ←─ proche ────→ "Salut"   (même sens)

Analogie classique :
  Roi - Homme + Femme ≈ Reine
```

> 🔍 **Elasticsearch** : Elasticsearch utilise les embeddings pour la recherche sémantique. Chercher *"appareils mobiles"* retrouve aussi *"téléphones, smartphones"* car leurs embeddings sont proches.

In [ ]:
# ============================================================
# EMBEDDINGS AVEC BERT
# ============================================================

if TRANSFORMERS_AVAILABLE:
    print("🔢 Extraction d'embeddings avec DistilBERT")
    print("=" * 65)

    model_embed = AutoModel.from_pretrained(MODEL_NAME)
    model_embed.eval()

    def get_sentence_embedding(text: str) -> np.ndarray:
        """Obtenir l'embedding d'une phrase (mean pooling)."""
        inputs = tokenizer(
            text, return_tensors='pt',
            padding=True, truncation=True, max_length=64
        )
        with torch.no_grad():
            outputs = model_embed(**inputs)
        # Mean pooling : moyenne sur tous les tokens
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        return embedding

    # Phrases de test
    test_sents = [
        "Excellent produit, très satisfait !",
        "Super article, je suis ravi !",
        "Très déçu, mauvaise qualité.",
        "Produit affreux, complètement raté.",
        "La météo est belle aujourd'hui."
    ]
    sent_labels = ["POSITIF", "POSITIF", "NÉGATIF", "NÉGATIF", "NEUTRE"]

    embeddings_list = [get_sentence_embedding(s) for s in test_sents]

    print(f"  Dimension d'un embedding : {embeddings_list[0].shape[0]}")
    print()

    # Calculer la similarité cosinus entre phrases
    def cosine_similarity(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    print("  Similarité cosinus entre phrases :")
    print(f"  {'Phrase 1':35s} | {'Phrase 2':35s} | Sim")
    print("  " + "-" * 85)

    pairs = [(0,1), (0,2), (2,3), (0,4), (2,4)]
    for i, j in pairs:
        sim = cosine_similarity(embeddings_list[i], embeddings_list[j])
        s1  = test_sents[i][:35]
        s2  = test_sents[j][:35]
        bar = '█' * int(sim * 15)
        print(f"  {s1:35s} | {s2:35s} | {sim:.3f} {bar}")

    print()
    print("  💡 Les phrases positives ont une similarité plus élevée entre elles")
else:
    print("⚠️  Transformers non disponible")

In [ ]:
# ============================================================
# VISUALISATION DES EMBEDDINGS (PCA 2D)
# ============================================================

if TRANSFORMERS_AVAILABLE:
    from sklearn.decomposition import PCA

    emb_matrix = np.array(embeddings_list)

    # Réduire à 2D avec PCA
    pca = PCA(n_components=2, random_state=SEED)
    emb_2d = pca.fit_transform(emb_matrix)

    fig, ax = plt.subplots(figsize=(10, 7))

    color_map = {"POSITIF": "#2ecc71", "NÉGATIF": "#e74c3c", "NEUTRE": "#95a5a6"}

    for i, (point, label, sent) in enumerate(zip(emb_2d, sent_labels, test_sents)):
        color = color_map[label]
        ax.scatter(point[0], point[1], c=color, s=200, zorder=5,
                   edgecolors='white', linewidths=2)
        ax.annotate(
            f"[{label}]\n{sent[:30]}...",
            xy=(point[0], point[1]),
            xytext=(point[0] + 0.02, point[1] + 0.02),
            fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3)
        )

    # Légende
    legend_patches = [
        mpatches.Patch(color="#2ecc71", label='POSITIF'),
        mpatches.Patch(color="#e74c3c", label='NÉGATIF'),
        mpatches.Patch(color="#95a5a6", label='NEUTRE')
    ]
    ax.legend(handles=legend_patches, loc='best')

    ax.set_title('🔢 Embeddings BERT — Visualisation PCA 2D\n'
                 '(Les phrases similaires sont proches dans l\'espace)', fontsize=12)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('embeddings_pca.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("📸 Graphique sauvegardé : embeddings_pca.png")
else:
    print("⚠️  Transformers non disponible — visualisation impossible")

<a id='5'></a>
## 5. 🤖 Transformers — BERT vs GPT

Les **Transformers** sont l'état de l'art en NLP depuis 2018 (*Attention is All You Need*, Vaswani et al.).

### Architecture Transformer

```
BERT (Encoder — Bidirectionnel)    GPT (Decoder — Autorégressif)
─────────────────────────────      ──────────────────────────────
Input: [CLS] texte [SEP]           Input: texte...
         │                                  │
   Self-Attention                    Masked Self-Attention
   (regarde gauche ET droite)        (regarde seulement à gauche)
         │                                  │
   Representations                  Génère le token suivant
   contextuelles                            │
         │                           → "... le prochain mot"
   Classification                   
   NER, QA...
```

### Comparaison BERT vs GPT

| Aspect | BERT | GPT |
|---|---|---|
| **Architecture** | Encoder (bidirectionnel) | Decoder (autorégressif) |
| **Entraînement** | Masquer des mots, les prédire | Prédire le mot suivant |
| **Cas d'usage** | Classification, NER, QA | Génération de texte |
| **Exemples** | DistilBERT, RoBERTa, CamemBERT | GPT-3, GPT-4, LLaMA |
| **Taille** | 66M–340M params | 117M–540B params |

### Pourquoi fine-tuner plutôt qu'entraîner from scratch ?

```
Entraîner BERT from scratch :
  → 3 semaines de calcul
  → 16+ GPU A100
  → Des milliers de GBs de données
  → Coût : ~500K$

Fine-tuner DistilBERT (notre approche) :
  → Quelques heures (voire minutes)
  → 1 GPU (ou même CPU)
  → Quelques centaines d'exemples
  → Coût : ~0$ (open-source)
```

> 📱 **WhatsApp, Telegram** : Les apps de messagerie utilisent le fine-tuning de BERT pour détecter les spams en temps réel. Chaque langue a son modèle fine-tuné.

<a id='6'></a>
## 6. 🧪 TP 4.2 — Fine-tuning BERT pour Sentiment Analysis

### Objectif
Fine-tuner **DistilBERT multilingue** sur notre dataset de 12 phrases (positif/négatif) et l'évaluer sur de nouvelles phrases.

In [ ]:
# ============================================================
# TP 4.2 — FINE-TUNING DISTILBERT
# ============================================================

if TRANSFORMERS_AVAILABLE:
    import mlflow
    import mlflow.sklearn

    print("🤗 Fine-tuning DistilBERT — Sentiment Analysis")
    print("=" * 65)

    # ---- Dataset étendu ----
    TRAIN_TEXTS = [
        "Excellent produit, très satisfait de mon achat !",
        "Mauvaise qualité, complètement déçu.",
        "Service client irréprochable, livraison rapide.",
        "Produit défectueux reçu. Remboursement refusé.",
        "Rapport qualité-prix parfait. Je recommande.",
        "Délai de livraison trop long, emballage abîmé.",
        "Fonctionne exactement comme décrit. Bonne surprise.",
        "Commande annulée sans explication. Service désastreux.",
        "Qualité premium, finition parfaite. J'adore !",
        "Rien ne correspond à la description. Arnaque.",
        "Très bon produit pour le prix.",
        "Incroyable ! Meilleur achat de l'année.",
        "Totalement inutile et mal conçu.",
        "Livraison express, produit conforme. Parfait !",
        "Pièce manquante dans le colis. Très décevant.",
        "Super rapport qualité-prix, je rachèterai."
    ]
    TRAIN_LABELS = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1]

    TEST_TEXTS = [
        "Vraiment déçu par cet achat, ne fonctionne pas.",
        "Parfait ! Exactement ce que je cherchais.",
        "Qualité médiocre, ne vaut pas son prix.",
        "Excellent, je recommande à tous mes amis !"
    ]
    TEST_LABELS = [0, 1, 0, 1]

    print(f"  Train set : {len(TRAIN_TEXTS)} phrases")
    print(f"  Test set  : {len(TEST_TEXTS)} phrases")
    print(f"  Classes   : 0=Négatif, 1=Positif")
    print()

    # ---- Tokeniser le dataset ----
    def tokenize_batch(texts, labels, max_length=64):
        encoded = tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoded['input_ids'],
            'attention_mask' : encoded['attention_mask'],
            'labels'         : torch.tensor(labels)
        }

    train_encodings = tokenize_batch(TRAIN_TEXTS, TRAIN_LABELS)
    test_encodings  = tokenize_batch(TEST_TEXTS,  TEST_LABELS)

    # ---- Créer un Dataset PyTorch ----
    class SentimentDataset(torch.utils.data.Dataset):
        def __init__(self, encodings):
            self.encodings = encodings

        def __len__(self):
            return len(self.encodings['labels'])

        def __getitem__(self, idx):
            return {
                'input_ids'      : self.encodings['input_ids'][idx],
                'attention_mask' : self.encodings['attention_mask'][idx],
                'labels'         : self.encodings['labels'][idx]
            }

    train_dataset = SentimentDataset(train_encodings)
    test_dataset  = SentimentDataset(test_encodings)

    print("✅ Datasets créés")
    print(f"   Train: {len(train_dataset)} samples")
    print(f"   Test : {len(test_dataset)} samples")
else:
    print("⚠️  Transformers non disponible")

In [ ]:
# ============================================================
# ENTRAÎNEMENT DU MODÈLE + MLFLOW TRACKING
# ============================================================

if TRANSFORMERS_AVAILABLE:
    from sklearn.metrics import accuracy_score, f1_score

    # ---- Charger DistilBERT pour classification ----
    print("🔄 Chargement de DistilBERT...")
    sentiment_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={0: "NÉGATIF", 1: "POSITIF"},
        label2id={"NÉGATIF": 0, "POSITIF": 1}
    )
    print(f"✅ Modèle chargé : {MODEL_NAME}")
    print(f"   Paramètres : {sum(p.numel() for p in sentiment_model.parameters()):,}")
    print()

    # ---- Configuration du fine-tuning ----
    training_args = TrainingArguments(
        output_dir         = './results_sentiment',
        num_train_epochs   = 3,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 4,
        learning_rate      = 2e-5,
        weight_decay       = 0.01,
        evaluation_strategy = 'epoch',
        save_strategy      = 'epoch',
        load_best_model_at_end = True,
        report_to          = 'none',
        logging_steps      = 10,
        seed               = SEED
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        acc = accuracy_score(labels, predictions)
        f1  = f1_score(labels, predictions, average='weighted')
        return {'accuracy': acc, 'f1': f1}

    trainer = Trainer(
        model           = sentiment_model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = test_dataset,
        compute_metrics = compute_metrics
    )

    # ---- Fine-tuning avec MLflow Tracking ----
    mlflow.set_tracking_uri('mlruns')
    mlflow.set_experiment('nlp_sentiment_analysis')

    with mlflow.start_run(run_name='distilbert_sentiment_v1'):
        mlflow.log_param('model_name',   MODEL_NAME)
        mlflow.log_param('n_epochs',     3)
        mlflow.log_param('learning_rate', 2e-5)
        mlflow.log_param('train_size',   len(TRAIN_TEXTS))
        mlflow.log_param('max_length',   64)

        print("🚀 Début du fine-tuning (3 epochs)...")
        train_result = trainer.train()

        # Évaluation finale
        eval_result = trainer.evaluate()

        mlflow.log_metric('eval_accuracy', eval_result['eval_accuracy'])
        mlflow.log_metric('eval_f1',       eval_result['eval_f1'])
        mlflow.log_metric('train_loss',    train_result.training_loss)

        print()
        print(f"  ✅ Fine-tuning terminé !")
        print(f"  📊 Accuracy  : {eval_result['eval_accuracy']:.4f}")
        print(f"  📊 F1-Score  : {eval_result['eval_f1']:.4f}")
        print(f"  📊 Train Loss: {train_result.training_loss:.4f}")

    # Sauvegarder le modèle
    os.makedirs('saved_model', exist_ok=True)
    sentiment_model.save_pretrained('saved_model/sentiment_bert')
    tokenizer.save_pretrained('saved_model/sentiment_bert')
    print()
    print("  💾 Modèle sauvegardé dans : saved_model/sentiment_bert/")
else:
    print("⚠️  Transformers non disponible")

In [ ]:
# ============================================================
# PRÉDICTIONS AVEC LE MODÈLE FINE-TUNÉ
# ============================================================

if TRANSFORMERS_AVAILABLE:
    print("🔮 Prédictions avec le modèle fine-tuné")
    print("=" * 65)

    # Utiliser le pipeline Hugging Face pour simplifier
    sentiment_pipeline = pipeline(
        'text-classification',
        model=sentiment_model,
        tokenizer=tokenizer,
        return_all_scores=True
    )

    new_phrases = [
        "Absolument ravi de cet achat, parfait !",
        "Produit inutile, perte d'argent.",
        "Livraison rapide, qualité au rendez-vous.",
        "Très déçu, rien ne fonctionne.",
        "Je recommande chaudement ce vendeur !",
        "Horrible expérience, évitez ce produit."
    ]
    expected = [1, 0, 1, 0, 1, 0]

    print(f"  {'Phrase':45s} {'Prédit':10s} {'Conf.':7s} {'Réel':8s} {'OK?'}")
    print("  " + "-" * 80)

    correct = 0
    for phrase, exp in zip(new_phrases, expected):
        scores = sentiment_pipeline(phrase[:512])[0]
        best   = max(scores, key=lambda x: x['score'])
        label  = 1 if best['label'] == 'POSITIF' else 0
        conf   = best['score']

        ok = "✅" if label == exp else "❌"
        if label == exp:
            correct += 1

        pred_str = "POS 😊" if label == 1 else "NEG 😞"
        exp_str  = "POS"    if exp == 1   else "NEG"
        print(f"  {phrase[:45]:45s} {pred_str:10s} {conf:.3f}  {exp_str:8s} {ok}")

    print()
    print(f"  🎯 Accuracy sur nouvelles phrases : {correct}/{len(new_phrases)} ({correct/len(new_phrases)*100:.0f}%)")
else:
    print("⚠️  Transformers non disponible")

<a id='7'></a>
## 7. 🏷️ Named Entity Recognition (NER)

**NER** = identifier les **entités nommées** dans un texte (personnes, lieux, organisations, dates, montants...).

### Types d'entités standard

| Tag | Type | Exemple |
|---|---|---|
| `PER` | Personne | Jean Dupont, Marie Curie |
| `ORG` | Organisation | Google, Hugging Face, ONU |
| `LOC` | Lieu | Paris, France, Alpes |
| `DATE` | Date/Temps | 2024, lundi, en janvier |
| `MONEY` | Montant | 500€, 1 million de dollars |
| `MISC` | Divers | Airbus A320, iPhone 15 |

> 🏥 **Extraction médicale** : NER extrait automatiquement les médicaments, doses, symptômes des dossiers patients. Économise des heures de transcription manuelle.

<a id='8'></a>
## 8. 🧪 TP 4.3 — NER avec Hugging Face

In [ ]:
# ============================================================
# TP 4.3 — NER AVEC HUGGING FACE PIPELINE
# ============================================================

if TRANSFORMERS_AVAILABLE:
    print("🏷️  Named Entity Recognition (NER) — TP 4.3")
    print("=" * 65)

    # NER multilingue
    ner_pipeline = pipeline(
        'ner',
        model='Davlan/distilbert-base-multilingual-cased-ner-hrl',
        aggregation_strategy='simple'
    )

    # Textes de test couvrant différents domaines
    ner_texts = [
        "Jean Dupont travaille chez Google à Paris depuis janvier 2020.",
        "Le président Emmanuel Macron a rencontré Angela Merkel à Berlin.",
        "Airbus a livré 50 A320 à Air France pour un montant de 5 milliards d'euros.",
        "L'OMS a publié un rapport sur le COVID-19 à Genève en mars 2020.",
        "Marie Curie a reçu le Prix Nobel de Chimie en 1911 à Stockholm."
    ]

    for text in ner_texts:
        entities = ner_pipeline(text)

        print(f"\n  📄 Texte : {text}")
        if entities:
            print(f"  Entités extraites :")
            for ent in entities:
                print(f"    [{ent['entity_group']:6s}] '{ent['word']:30s}' (conf: {ent['score']:.3f})")
        else:
            print("  Aucune entité détectée")
else:
    print("⚠️  Transformers non disponible")
    print()
    # Simulation NER manuelle avec regex
    print("🔄 Simulation NER avec règles (fallback sans Transformers)")
    print("-" * 65)

    def simple_ner_rules(text):
        """NER basé sur des règles simples (demo sans transformers)."""
        entities = []
        patterns = [
            (r'\b(Paris|Berlin|Lyon|Genève|Stockholm)\b', 'LOC'),
            (r'\b(Google|Airbus|Air France|OMS|ONU)\b', 'ORG'),
            (r'\b(20\d{2})\b', 'DATE'),
            (r'\b(janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)\b', 'DATE'),
            (r'\b[A-Z][a-z]+ [A-Z][a-z]+\b', 'PER'),
        ]
        for pattern, label in patterns:
            for match in re.finditer(pattern, text):
                entities.append({'word': match.group(), 'entity_group': label})
        return entities

    ner_texts = [
        "Jean Dupont travaille chez Google à Paris depuis janvier 2020.",
        "Marie Curie a reçu le Prix Nobel en 1911 à Stockholm."
    ]
    for text in ner_texts:
        ents = simple_ner_rules(text)
        print(f"\n  📄 {text}")
        for e in ents:
            print(f"    [{e['entity_group']:6s}] '{e['word']}'")

In [ ]:
# ============================================================
# ÉVALUATION DU NER — Precision, Recall, F1
# ============================================================

print("📊 Évaluation du NER — Métriques")
print("=" * 65)

# Texte annoté manuellement (gold standard)
gold_standard = [
    {
        "text"    : "Jean Dupont travaille chez Google à Paris.",
        "entities": [("Jean Dupont", "PER"), ("Google", "ORG"), ("Paris", "LOC")]
    },
    {
        "text"    : "Marie Curie a reçu le Prix Nobel en 1911 à Stockholm.",
        "entities": [("Marie Curie", "PER"), ("1911", "DATE"), ("Stockholm", "LOC")]
    },
    {
        "text"    : "Airbus a livré des A320 à Air France.",
        "entities": [("Airbus", "ORG"), ("Air France", "ORG")]
    }
]

def simple_ner_rules(text):
    entities = []
    patterns = [
        (r'\b(Paris|Berlin|Lyon|Genève|Stockholm)\b', 'LOC'),
        (r'\b(Google|Airbus|Air France|OMS)\b',       'ORG'),
        (r'\b(1[89]\d{2}|20\d{2})\b',                'DATE'),
        (r'\b[A-Z][a-z]+ [A-Z][a-z]+\b',             'PER'),
    ]
    for pattern, label in patterns:
        for match in re.finditer(pattern, text):
            entities.append((match.group(), label))
    return entities

total_tp = total_fp = total_fn = 0

for sample in gold_standard:
    gold_ents = set(sample['entities'])
    pred_ents = set(simple_ner_rules(sample['text']))

    tp = len(gold_ents & pred_ents)
    fp = len(pred_ents - gold_ents)
    fn = len(gold_ents - pred_ents)

    total_tp += tp
    total_fp += fp
    total_fn += fn

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"  Texte : {sample['text'][:50]}...")
    print(f"    Gold : {gold_ents}")
    print(f"    Pred : {pred_ents}")
    print(f"    TP={tp}, FP={fp}, FN={fn} | Precision={prec:.2f}, Recall={rec:.2f}")
    print()

# Métriques globales
precision_global = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall_global    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1_global        = 2 * precision_global * recall_global / (precision_global + recall_global + 1e-9)

print("=" * 65)
print(f"  📊 Métriques globales NER (système basé sur règles) :")
print(f"     Precision : {precision_global:.4f}")
print(f"     Recall    : {recall_global:.4f}")
print(f"     F1-Score  : {f1_global:.4f}")
print()
print("  💡 Un modèle NER BERT obtiendrait F1 > 0.90 sur ce même dataset")

<a id='9'></a>
## 9. ❓ Question Answering (QA)

**QA** = donner un **contexte** et une **question**, le modèle trouve la **réponse** dans le contexte.

> 🤖 **Chatbots bancaires** : Les chatbots utilisent QA pour répondre automatiquement aux questions des clients sur leurs comptes, virements, frais.

In [ ]:
# ============================================================
# QUESTION ANSWERING (QA)
# ============================================================

if TRANSFORMERS_AVAILABLE:
    print("❓ Question Answering — Pipeline Hugging Face")
    print("=" * 65)

    qa_pipeline = pipeline(
        'question-answering',
        model='distilbert-base-cased-distilled-squad'
    )

    qa_samples = [
        {
            "context"  : "Paris est la capitale de la France. Elle est connue pour la Tour Eiffel, le Louvre et ses magnifiques ponts. La Seine traverse la ville.",
            "question" : "Quel fleuve traverse Paris ?"
        },
        {
            "context"  : "MLflow est un framework open-source créé par Databricks en 2018. Il permet de tracker les expériences ML, gérer les modèles et les déployer en production.",
            "question" : "Qui a créé MLflow ?"
        },
        {
            "context"  : "Apache Airflow est un outil d'orchestration de workflows créé par Airbnb en 2014. Il utilise des DAGs (Directed Acyclic Graphs) pour définir les pipelines.",
            "question" : "Quelle entreprise a créé Apache Airflow ?"
        }
    ]

    for i, sample in enumerate(qa_samples):
        result = qa_pipeline(question=sample['question'], context=sample['context'])
        print(f"  Exemple {i+1}:")
        print(f"    Question  : {sample['question']}")
        print(f"    Réponse   : {result['answer']}")
        print(f"    Confiance : {result['score']:.4f}")
        print(f"    Position  : caractères {result['start']}-{result['end']}")
        print()

else:
    # Simulation QA sans transformers
    print("🔄 Simulation QA (fallback sans Transformers)")
    print("-" * 65)
    context = "MLflow est un framework open-source créé par Databricks en 2018."
    question = "Qui a créé MLflow ?"

    # Extraction simple par mots-clés
    keywords = [w for w in question.lower().split() if len(w) > 3 and w not in ['créé', 'qui', 'quel', 'est']]
    sentences = context.split('.')
    best_sent = max(sentences, key=lambda s: sum(k in s.lower() for k in keywords))

    print(f"  Contexte  : {context}")
    print(f"  Question  : {question}")
    print(f"  Réponse approx : {best_sent.strip()}")

<a id='10'></a>
## 10. 🧪 TP 4.4 — API Flask pour Modèle NLP

### Objectif
Créer une **API Flask de production** pour le modèle de sentiment analysis, avec endpoints `/health`, `/predict` et `/batch_predict`.

In [ ]:
# ============================================================
# TP 4.4 — GÉNÉRER L'API FLASK NLP
# ============================================================

os.makedirs('nlp_api', exist_ok=True)

flask_nlp_app = '''# app_nlp.py — API Flask pour Sentiment Analysis NLP
# M1MLOP — Amadou MAIGA — TP 4.4
#
# Lancer   : python app_nlp.py
# Tester   : curl -X POST http://localhost:5000/predict \\
#             -H "Content-Type: application/json" \\
#             -d \'{"text": "Excellent produit, très satisfait !"}\'\n
from flask import Flask, request, jsonify
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import logging
import time
from datetime import datetime
import torch

app = Flask(__name__)
logging.basicConfig(
    level=logging.INFO,
    format=\'%(asctime)s - %(name)s - %(levelname)s - %(message)s\'
)
logger = logging.getLogger(__name__)

# ===== CHARGEMENT DU MODÈLE AU DÉMARRAGE (une seule fois) =====
MODEL_PATH     = \'saved_model/sentiment_bert\'
MODEL_VERSION  = \'1.0\'
MAX_TEXT_LENGTH = 512

logger.info("Loading NLP model...")
start_load = time.time()

try:
    classifier = pipeline(
        \'text-classification\',
        model=MODEL_PATH,
        return_all_scores=True
    )
    load_time = time.time() - start_load
    logger.info(f"Model loaded in {load_time:.2f}s")
except Exception as e:
    logger.error(f"Failed to load model: {e}")
    classifier = None

# ===== MÉTRIQUES INTERNES =====
metrics = {
    \'total_predictions\': 0,
    \'total_errors\'     : 0,
    \'latencies\'        : [],
    \'started_at\'       : datetime.utcnow().isoformat()
}

# ===== ENDPOINTS =====

@app.route(\'/health\', methods=[\'GET\'])
def health():
    """Health check — utilisé par K8s liveness/readiness probe."""
    return jsonify({
        \'status\'        : \'healthy\' if classifier else \'unhealthy\',
        \'model_version\'  : MODEL_VERSION,
        \'model_loaded\'   : classifier is not None,
        \'timestamp\'      : datetime.utcnow().isoformat()
    }), 200 if classifier else 503

@app.route(\'/info\', methods=[\'GET\'])
def info():
    """Informations sur le modèle et les métriques."""
    avg_latency = (sum(metrics[\'latencies\']) / len(metrics[\'latencies\']) * 1000
                   if metrics[\'latencies\'] else 0)
    return jsonify({
        \'model_version\'       : MODEL_VERSION,
        \'task\'                : \'sentiment-analysis\',
        \'labels\'              : [\'NÉGATIF\', \'POSITIF\'],
        \'total_predictions\'   : metrics[\'total_predictions\'],
        \'total_errors\'        : metrics[\'total_errors\'],
        \'avg_latency_ms\'      : round(avg_latency, 2),
        \'uptime\'              : metrics[\'started_at\']
    })

@app.route(\'/predict\', methods=[\'POST\'])
def predict():
    """
    Prédire le sentiment d\'une phrase.

    Input  : {"text": "Le produit est excellent !"}
    Output : {"label": "POSITIF", "score": 0.98, "model_version": "1.0"}
    """
    if not classifier:
        return jsonify({\'error\': \'Model not loaded\'}), 503

    start = time.time()

    try:
        data = request.get_json()
        if not data or \'text\' not in data:
            return jsonify({\'error\': \'Missing field: text\'}), 400

        text = str(data[\'text\']).strip()
        if not text:
            return jsonify({\'error\': \'Text cannot be empty\'}), 400

        # Tronquer si trop long
        text = text[:MAX_TEXT_LENGTH]

        # Prédiction
        scores  = classifier(text)[0]
        best    = max(scores, key=lambda x: x[\'score\'])
        all_scores = {s[\'label\']: round(float(s[\'score\']), 4) for s in scores}

        latency = time.time() - start
        metrics[\'total_predictions\'] += 1
        metrics[\'latencies\'].append(latency)

        logger.info(f"Predicted: {best[\'label\']} ({best[\'score\']*100:.1f}%) in {latency*1000:.1f}ms")

        return jsonify({
            \'text\'          : text[:100],
            \'label\'         : best[\'label\'],
            \'score\'         : round(float(best[\'score\']), 4),
            \'all_scores\'    : all_scores,
            \'model_version\'  : MODEL_VERSION,
            \'latency_ms\'    : round(latency * 1000, 2),
            \'timestamp\'     : datetime.utcnow().isoformat()
        })

    except Exception as e:
        metrics[\'total_errors\'] += 1
        logger.error(f"Prediction error: {str(e)}")
        return jsonify({\'error\': str(e)}), 500

@app.route(\'/batch_predict\', methods=[\'POST\'])
def batch_predict():
    """
    Prédire sur plusieurs textes en une seule requête (optimisé).

    Input  : {"texts": ["texte 1", "texte 2", ...]}
    Output : {"predictions": [{"label": ..., "score": ...}, ...]}
    """
    if not classifier:
        return jsonify({\'error\': \'Model not loaded\'}), 503

    try:
        data  = request.get_json()
        texts = data.get(\'texts\', [])

        if not texts or not isinstance(texts, list):
            return jsonify({\'error\': \'Provide a list of texts\'}), 400

        texts = [str(t)[:MAX_TEXT_LENGTH] for t in texts[:100]]  # Max 100 textes

        start   = time.time()
        results = []

        for text in texts:
            scores = classifier(text)[0]
            best   = max(scores, key=lambda x: x[\'score\'])
            results.append({
                \'text\'  : text[:80],
                \'label\'  : best[\'label\'],
                \'score\'  : round(float(best[\'score\']), 4)
            })

        latency = time.time() - start
        metrics[\'total_predictions\'] += len(texts)

        return jsonify({
            \'predictions\' : results,
            \'count\'       : len(results),
            \'latency_ms\'  : round(latency * 1000, 2)
        })

    except Exception as e:
        return jsonify({\'error\': str(e)}), 500

if __name__ == \'__main__\':
    app.run(host=\'0.0.0.0\', port=5000, debug=False)
'''

dockerfile_nlp = '''# Dockerfile NLP API — M1MLOP — Amadou MAIGA
FROM python:3.10-slim

WORKDIR /app

# Dépendances système
RUN apt-get update && apt-get install -y curl && rm -rf /var/lib/apt/lists/*

# Dépendances Python
COPY requirements_nlp.txt .
RUN pip install --no-cache-dir -r requirements_nlp.txt

# Copier le code et le modèle
COPY app_nlp.py .
COPY saved_model/ ./saved_model/

EXPOSE 5000

HEALTHCHECK --interval=30s --timeout=15s --start-period=60s \\\
  CMD curl -f http://localhost:5000/health || exit 1

CMD ["python", "app_nlp.py"]
'''

requirements_nlp = '''flask==3.0.0
transformers==4.35.0
torch==2.1.0
numpy==1.24.0
'''

with open('nlp_api/app_nlp.py', 'w') as f:
    f.write(flask_nlp_app)
with open('nlp_api/Dockerfile', 'w') as f:
    f.write(dockerfile_nlp)
with open('nlp_api/requirements_nlp.txt', 'w') as f:
    f.write(requirements_nlp)

print("✅ Fichiers API NLP générés :")
print("   nlp_api/app_nlp.py            (API Flask complète)")
print("   nlp_api/Dockerfile            (conteneurisation)")
print("   nlp_api/requirements_nlp.txt  (dépendances)")
print()
print("📋 Endpoints disponibles :")
print("   GET  /health            → Status du service")
print("   GET  /info              → Infos modèle + métriques")
print("   POST /predict           → Prédire un seul texte")
print("   POST /batch_predict     → Prédire plusieurs textes")
print()
print("📋 Commandes Docker :")
print("   docker build -t nlp-api:1.0 nlp_api/")
print("   docker run -p 5000:5000 nlp-api:1.0")

<a id='11'></a>
## 11. 🔄 MLOps appliqué au NLP

Les techniques MLOps des jours 1-3 s'appliquent aussi aux modèles de langage :

| MLOps Jour 1-3 | Appliqué au NLP |
|---|---|
| MLflow tracking | Logger loss, accuracy, F1 du fine-tuning |
| Drift Detection | Détecter si la distribution des sentiments change |
| A/B Testing | Comparer BERT v1 vs BERT v2 sur 50% utilisateurs |
| Explicabilité | Quels tokens ont influencé la prédiction |
| Fairness | Le modèle est-il biaisé selon la langue ? |

In [ ]:
# ============================================================
# DRIFT DETECTION SUR DONNÉES NLP
# ============================================================
from scipy.stats import ks_2samp

print("🌊 Drift Detection sur données NLP")
print("=" * 65)
print()
print("Scénario : Le modèle de sentiment est en production.")
print("On compare la distribution des scores en janvier vs mars.")
print()

np.random.seed(SEED)

# Janvier : clients satisfaits (produit bien lancé)
scores_janvier = np.random.beta(8, 2, 500)   # distribution skewed positive
# Mars : clients moins satisfaits (problèmes qualité)
scores_mars    = np.random.beta(4, 5, 500)   # distribution plus neutre

_, p_val = ks_2samp(scores_janvier, scores_mars)

print(f"  Distribution des scores de confiance POSITIF :")
print(f"  Janvier : mean={scores_janvier.mean():.3f}, std={scores_janvier.std():.3f}")
print(f"  Mars    : mean={scores_mars.mean():.3f}, std={scores_mars.std():.3f}")
print(f"  KS test p-value : {p_val:.6f}")
print()

if p_val < 0.05:
    print("  🚨 DRIFT DÉTECTÉ : Les sentiments des utilisateurs ont changé !")
    print("  👉 Action : Analyser les nouvelles données, retrainer si nécessaire")
else:
    print("  ✅ Pas de drift détecté.")

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores_janvier, bins=40, alpha=0.7, color='green',  label='Janvier (positif dominant)', density=True)
ax.hist(scores_mars,    bins=40, alpha=0.7, color='orange', label='Mars (plus mitigé)',          density=True)
ax.set_title(f'🌊 Drift NLP — Distribution des scores de sentiment\np-value KS = {p_val:.4f} {"→ DRIFT DÉTECTÉ 🚨" if p_val < 0.05 else "→ OK ✅"}')
ax.set_xlabel('Score de confiance POSITIF')
ax.set_ylabel('Densité')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('nlp_drift_detection.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Graphique sauvegardé : nlp_drift_detection.png")

In [ ]:
# ============================================================
# EXPLICABILITÉ NLP — IMPORTANCE DES TOKENS
# ============================================================

print("⚖️  Explicabilité NLP — Importance des tokens")
print("=" * 65)
print()

# Simulation de l'importance des tokens (LIME-like)
# En production : utiliser captum ou SHAP text

def estimate_token_importance(text, label_fn, tokenizer_fn, n_samples=20):
    """
    Estime l'importance de chaque token par perturbation (approche LIME simplifiée).
    Retire un token à la fois et mesure l'impact sur la prédiction.
    """
    tokens = text.split()
    base_score = label_fn(text)
    importances = []

    for i, token in enumerate(tokens):
        # Retirer le token
        modified = " ".join([t for j, t in enumerate(tokens) if j != i])
        modified_score = label_fn(modified) if modified.strip() else 0.5
        # Importance = différence de score quand le token est retiré
        importance = base_score - modified_score
        importances.append((token, importance))

    return importances

# Simuler un score de sentiment simple (heuristique)
POSITIVE_WORDS = {'excellent','parfait','ravi','super','génial','recommande','satisfait','bravo','bien','bon'}
NEGATIVE_WORDS = {'déçu','mauvais','nul','problème','défectueux','scandaleux','horrible','terrible','annulée'}

def heuristic_sentiment(text):
    words = set(text.lower().split())
    pos = sum(1 for w in words if w in POSITIVE_WORDS)
    neg = sum(1 for w in words if w in NEGATIVE_WORDS)
    total = pos + neg
    return pos / total if total > 0 else 0.5

test_explanations = [
    "Excellent produit, très satisfait de mon achat !",
    "Mauvaise qualité, complètement déçu du service."
]

for text in test_explanations:
    score = heuristic_sentiment(text)
    label = "POSITIF 😊" if score > 0.5 else "NÉGATIF 😞"
    importances = estimate_token_importance(text, heuristic_sentiment, None)

    print(f"  📄 Texte   : {text}")
    print(f"  Prédiction : {label} (score={score:.3f})")
    print(f"  Importance des tokens :")

    importances_sorted = sorted(importances, key=lambda x: abs(x[1]), reverse=True)
    for token, imp in importances_sorted[:5]:
        direction = "⬆️" if imp > 0.01 else "⬇️" if imp < -0.01 else "➡️"
        bar       = '█' * int(abs(imp) * 20)
        print(f"    {direction} '{token:20s}' impact={imp:+.3f}  {bar}")
    print()

<a id='12'></a>
## 12. 📝 Résumé et Bilan du Jour 4

### ✅ Ce que tu as fait aujourd'hui

| Concept | Ce que tu as réalisé |
|---|---|
| **Preprocessing NLP** | Pipeline complet : lowercase, ponctuation, stopwords, tokenisation |
| **BoW + TF-IDF** | Vectorisation classique + classification Naive Bayes |
| **Tokeniser HF** | Subword tokenization avec DistilBERT, 119K+ vocab |
| **Embeddings** | Similarité cosinus, visualisation PCA 2D |
| **Fine-tuning BERT** | DistilBERT multilingue fine-tuné, MLflow tracking |
| **NER** | Extraction d'entités + évaluation Precision/Recall/F1 |
| **QA** | Question Answering sur contextes structurés |
| **API Flask NLP** | 4 endpoints + Dockerfile + batch predict |
| **MLOps NLP** | Drift sur scores NLP + explicabilité tokens |

### 🎯 Points clés

1. **Tokenisation** et **embeddings** convertissent le texte en vecteurs numériques
2. **BERT** est bidirectionnel (classification), **GPT** est autorégressif (génération)
3. **Fine-tuning** adapte un modèle pré-entraîné en quelques heures vs. semaines
4. **NER**, **sentiment analysis**, **QA** sont des applications pratiques immédiates
5. Les techniques **MLOps** (drift, fairness, explicabilité) s'appliquent aussi au NLP

### 🚀 Demain — Jour 5
**Intégration complète + Projet Final** :  
→ Pipeline Airflow + FastAPI + Kubernetes + Monitoring  
→ Système de classification de tickets support

---

### 📁 Fichiers générés
```
jour4/
├── jour4_nlp_transformers.ipynb
├── saved_model/sentiment_bert/       ← modèle fine-tuné
├── nlp_api/
│   ├── app_nlp.py
│   ├── Dockerfile
│   └── requirements_nlp.txt
├── embeddings_pca.png
└── nlp_drift_detection.png
```

In [ ]:
# ============================================================
# ✅ VÉRIFICATION FINALE — Bilan du Jour 4
# ============================================================

print("=" * 65)
print("         🎉 BILAN JOUR 4 — NLP & TRANSFORMERS")
print("=" * 65)
print()

checks = [
    ("TP 4.1 — Preprocessing NLP",       True),
    ("TP 4.1 — BoW + TF-IDF",            True),
    ("TP 4.1 — Tokenizer Hugging Face",   TRANSFORMERS_AVAILABLE),
    ("TP 4.2 — Fine-tuning DistilBERT",   TRANSFORMERS_AVAILABLE and os.path.exists('saved_model/sentiment_bert')),
    ("TP 4.2 — MLflow tracking NLP",      TRANSFORMERS_AVAILABLE),
    ("TP 4.3 — NER extraction",           True),
    ("TP 4.3 — NER évaluation P/R/F1",    True),
    ("TP 4.4 — API Flask NLP générée",    os.path.exists('nlp_api/app_nlp.py')),
    ("TP 4.4 — Dockerfile NLP",           os.path.exists('nlp_api/Dockerfile')),
    ("MLOps NLP — Drift detection",       os.path.exists('nlp_drift_detection.png')),
    ("MLOps NLP — Token importance",      True),
]

all_ok = True
for label, condition in checks:
    status = "✅" if condition else "⚠️ "
    if not condition:
        all_ok = False
    print(f"  {status} {label}")

print()
print("=" * 65)
if all_ok:
    print("  🏆 Tous les TPs du Jour 4 sont COMPLÉTÉS !")
else:
    print("  ⚠️  Certains TPs nécessitent Transformers installé.")
    print("  💡 Lance la cellule d'installation et réexécute.")
print("=" * 65)
print()
print("  📌 Git commit suggéré :")
print("  git add jour4/")
print('  git commit -m "✅ Jour 4 - NLP Transformers + BERT Fine-tuning + NER + API"')
print("  git push")
print("=" * 65)